# Preparação dos Datasets

Este notebook parte do **dataset original** (`dataset_original.xlsx`) e produz **7 datasets separados**, um para cada área térmica do fogão.

O escopo deste notebook é intencionalmente limitado a duas operações:

1. **Padronizar categorias duplicadas** — unificar variações de grafia (`To back` / `To Back`), maiúsculas/minúsculas e sinônimos (`Broil Case 2` / `Broil C2`) que, se mantidos, seriam lidos como categorias diferentes.
2. **Dividir em 7 datasets** — um por `Key Area`, no formato *wide* (uma linha por ensaio, uma coluna por componente).

Nenhum outro tratamento é feito aqui. Outliers, valores nulos e codificação (one-hot) ficam para notebooks posteriores, para que cada etapa de limpeza seja documentada separadamente.

Quatro colunas do dataset original são descartadas: `Reason`, `TC Number` (ambos não têm impacto na temperatura - variável alvo), `Active Limit` e `Key Area`. A `Active Limit` não é apagada de vez, será salva à parte, como tabela de referência para a comparação pós-predição.


## 1. Configuração

No Google Colab, o jeito mais simples é subir o arquivo pela barra lateral (ícone de pasta) e ajustar o caminho abaixo.


In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2/colabs_atualizados"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import unicodedata

# No Colab, basta subir o arquivo pela barra lateral (icone de pasta):
# ele fica ao lado do notebook e o caminho relativo abaixo funciona.
CAMINHO_ORIGINAL = "dataset_original.xlsx"

# Fora do Colab, o arquivo vive em documents/outros no repositorio.
if not os.path.exists(CAMINHO_ORIGINAL):
    CAMINHO_ORIGINAL = os.path.join("..", "..", "..", "documents", "outros", "dataset_original.xlsx")

ABA = 0  # a planilha tem uma unica aba: "Option 3 - First cases HE"
print("Lendo de:", CAMINHO_ORIGINAL)

## 2. Leitura da planilha bruta

A planilha tem um formato **transposto** que não pode ser lido diretamente com `pd.read_excel` no modo padrão:

- **Colunas** (da 5ª em diante) são os ensaios — `Run 1`, `Run 2`, etc.
- **Linhas 2 a 9** descrevem a configuração de cada ensaio (Blower Direction, Housing, Run Case…).
- **Linhas 11 em diante** são os componentes medidos, e cada célula é a temperatura daquele componente naquele ensaio.

Por isso lemos sem cabeçalho (`header=None`) e reconstruímos a estrutura manualmente.


In [ ]:
raw = pd.read_excel(CAMINHO_ORIGINAL, sheet_name=ABA, header=None)
print("Formato bruto (linhas x colunas):", raw.shape)

raw.iloc[:12, :6] # printa da linha 0 até a 11 e da coluna 0 a 5


## 3. Extração das configurações de cada ensaio

As linhas 2 a 9 da planilha (índices 1 a 8) trazem as características de design de cada ensaio. A linha 1 (`Reason`) é descartada, pois é responsável por descrever o *motivo* do teste, e não se enquadra como uma variável física que influencia na temperatura.

Cada ensaio recebe um identificador interno `run_idx` (0, 1, 2…). Usamos esse índice em vez do nome (`Run 1`, `Run 2`…) porque os nomes se repetem na planilha e não serviriam como chave única.


In [ ]:
# Linha do Excel -> nome da feature. 'Reason' (linha 1) fica de fora: descartada.
LINHAS_FEATURES = {
    2: "blower_direction",
    3: "housing",
    4: "board_position",
    5: "run_case",
    6: "losses",
    7: "deflector_hole",
    8: "material_coil_support",
}

n_ensaios = raw.shape[1] - 4  # as 4 primeiras colunas são metadados dos componentes

ensaios = pd.DataFrame({"run_idx": range(n_ensaios)}) # cria uma nova tabela só com run_idx
for linha, nome in LINHAS_FEATURES.items():
    valores = raw.iloc[linha, 4:].tolist()
    ensaios[nome] = [str(v).strip() if pd.notna(v) else np.nan for v in valores]

print("Ensaios encontrados:", len(ensaios))
ensaios.head()


## 4. Padronização das categorias duplicadas

Esta é a primeira das duas operações centrais do notebook.

O problema: a mesma categoria aparece escrita de formas diferentes na planilha. `To back` e `To Back` são o mesmo valor, mas o computador os trata como distintos. O mesmo vale para sinônimos como `Broil Case 2` e `Broil C2`.

A padronização acontece em dois passos:

1. **Normalização geral** — tudo em minúsculas, sem espaços duplicados nem sobrando nas pontas. Isso já resolve a maioria das duplicatas (as que diferem só por grafia/maiúsculas).
2. **Mapa de sinônimos** — para casos em que a mesma coisa tem nomes diferentes, unificamos manualmente.

Os nulos são preservados como `NaN` — nenhum valor é inventado para preencher vazios.


In [ ]:
def normalizar(texto):
    """Minúsculas, sem acento, sem espaço duplicado ou sobrando nas pontas."""
    if pd.isna(texto):
        return np.nan
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto).strip().lower()

COLUNAS_CATEGORICAS = list(LINHAS_FEATURES.values())
for col in COLUNAS_CATEGORICAS:
    ensaios[col] = ensaios[col].map(normalizar)


In [ ]:
# Mapa de sinônimos: valores que significam a mesma coisa mas foram escritos diferente.
SINONIMOS = {
    "run_case": {
        "broil case 2": "broil c2",
        "broil qual": "broil qualitative",
        "qualitative": "broil qualitative",
    },
    "material_coil_support": {
        "aluminium plate": "aluminium",
    },
}

for coluna, mapa in SINONIMOS.items():
    ensaios[coluna] = ensaios[coluna].replace(mapa)

# Conferência: categorias finais de cada feature
for col in COLUNAS_CATEGORICAS:
    print(col, "->", sorted(ensaios[col].dropna().unique().tolist()))


> **Observação sobre `housing`.** Depois da padronização geral, `housing` ainda tem três valores: `closed`, `open` e `open - updated`. Neste notebook eles são mantidos como estão — a decisão de unificar `open - updated` com `open` é um tratamento posterior, fora do escopo desta etapa de padronização básica.

> **Observação sobre `losses`.** É a feature com mais categorias (13). Todas são mantidas; nenhuma é agrupada aqui.


## 5. Extração dos componentes e suas temperaturas

As linhas 11 em diante (índice 10+) trazem os componentes. As quatro primeiras colunas são: `TC Number`, `TC Name`, `Key Area` e `Active Limit`. As demais são as temperaturas por ensaio.

Dois cuidados na leitura:

- A planilha tem **linhas de seção** (ex.: "Mainback Components") sem `TC Name` — elas são ignoradas.
- Quatro componentes ("Aluminium Plate" FR/FL/RR/RL) têm `Key Area` em branco na planilha, mas pertencem a **Coil Temps**. Preenchemos o `Key Area` para baixo (`ffill`) para capturá-los na área correta.


In [ ]:
PRIMEIRA_LINHA_COMPONENTE = 10
ULTIMA_LINHA_COMPONENTE = 93  # exclusivo

# O corte precisa ser justificado: o que fica de fora e impresso antes do descarte.
descartadas = raw.iloc[ULTIMA_LINHA_COMPONENTE:]
print(f"Linhas ignoradas pelo corte ({ULTIMA_LINHA_COMPONENTE} a {len(raw) - 1}):", len(descartadas))
for pos, linha in descartadas.iterrows():
    n_valores = int(linha.iloc[4:].notna().sum())
    print(f"  linha {pos}: tc_name={linha.iloc[1]!r}, valores preenchidos={n_valores}")
print("Nenhuma delas possui tc_name, portanto nao seria aproveitada como componente.")
print()

componentes = raw.iloc[PRIMEIRA_LINHA_COMPONENTE:ULTIMA_LINHA_COMPONENTE].copy()
componentes.columns = ["tc_number", "tc_name", "key_area", "active_limit"] + list(range(n_ensaios))

# Remove linhas de secao (sem nome de componente)
componentes = componentes[componentes["tc_name"].notna()].copy()

# Key Area em branco herda a da linha de cima (caso das Aluminium Plate -> Coil Temps).
# O que foi herdado e registrado, para que a heranca nao ocorra em silencio.
herdou = componentes["key_area"].isna()
componentes["key_area"] = componentes["key_area"].ffill()
print("Componentes que herdaram key_area por ffill:", int(herdou.sum()))
if herdou.any():
    print(componentes.loc[herdou, ["tc_name", "key_area"]].to_string(index=False))
print()

# Limpa espacos duplicados nos nomes de componente
componentes["tc_name"] = (componentes["tc_name"].astype(str)
                          .str.replace(r"\s+", " ", regex=True).str.strip())

# Nomes repetidos dentro da mesma area seriam colapsados no pivot adiante.
duplicados = componentes[componentes.duplicated(["key_area", "tc_name"], keep=False)]
if len(duplicados):
    print("ATENCAO: tc_name repetido dentro da mesma key_area:")
    print(duplicados[["key_area", "tc_name"]].to_string(index=False))
else:
    print("Nenhum tc_name repetido dentro de uma mesma key_area.")
print()

print("Componentes encontrados:", len(componentes))
print("Areas:", sorted(componentes["key_area"].unique().tolist()))
componentes[["tc_number", "tc_name", "key_area", "active_limit"]].head()

## 6. Tabela de referência do `Active Limit`

O `Active Limit` não entra no treino, mas não é jogado fora: ele é o valor que, depois da predição, dirá se a temperatura prevista está em zona segura, no limite ou excedida. Guardamos numa tabela à parte — um limite por componente.


In [ ]:
limites = componentes[["tc_name", "key_area", "active_limit"]].copy()

# Extrai o valor numérico do limite quando existir (alguns são "Monitor", sem número)
limites["limite_num"] = pd.to_numeric(
    limites["active_limit"].astype(str).str.extract(r"(\d+\.?\d*)")[0],
    errors="coerce"
)
limites["apenas_monitorar"] = limites["limite_num"].isna()

limites.to_csv("limites_ativos.csv", index=False)
print("Tabela de limites salva: limites_ativos.csv")
limites.head()


## 7. Reorganização para o formato *wide* e junção com as configurações

Transformamos a matriz de temperaturas em formato longo (uma linha por par componente×ensaio) e juntamos com as configurações de cada ensaio. Em seguida, remontamos no formato **wide**: uma linha por ensaio, uma coluna por componente — o mesmo formato dos datasets de referência.

`TC Number` é descartado aqui (feedback: inútil para a predição, além de estar quebrado com `#REF!` na planilha original).


In [ ]:
# Wide -> longo (uma linha por componente medido em cada ensaio)
longo = componentes.melt(
    id_vars=["tc_name", "key_area"],
    value_vars=list(range(n_ensaios)),
    var_name="run_idx",
    value_name="temperatura_c",
)
longo["temperatura_c"] = pd.to_numeric(longo["temperatura_c"], errors="coerce")

# Junta as configurações de cada ensaio (via run_idx)
longo = longo.merge(ensaios, on="run_idx", how="left")

print("Formato longo:", longo.shape)
longo.head()


### 7.1 Diagnóstico da conversão numérica

A conversão das temperaturas é feita com errors igual a coerce, o que transforma em nulo qualquer célula que não seja numérica, sem emitir aviso. Um resíduo como #REF!, um traço ou uma nota de rodapé passaria a ser indistinguível de uma medição que nunca ocorreu.

A célula abaixo registra quantas células preenchidas foram perdidas nessa conversão e verifica se os valores resultantes estão dentro de faixa fisicamente plausível. A verificação de faixa cobre também a dúvida sobre as células do Run 5 exibidas como percentual: caso o valor houvesse sido digitado como 6003 em vez de 60,03, ele apareceria aqui como leitura acima de 300 graus Celsius.

In [ ]:
# Celulas que estavam preenchidas na planilha mas nao sobreviveram a conversao numerica
bruto = componentes[list(range(n_ensaios))]
preenchidas = int(bruto.notna().sum().sum())
numericas = int(bruto.apply(pd.to_numeric, errors="coerce").notna().sum().sum())
print(f"Celulas preenchidas na planilha: {preenchidas}")
print(f"Celulas convertidas em numero:   {numericas}")
print(f"Perdidas na conversao:           {preenchidas - numericas}")

if preenchidas != numericas:
    nao_num = bruto[bruto.notna() & bruto.apply(pd.to_numeric, errors="coerce").isna()]
    print("Conteudo nao numerico encontrado:")
    print(pd.unique(nao_num.values.ravel())[:20])

# Faixa fisicamente plausivel para temperatura medida em fogao em operacao
MIN_PLAUSIVEL, MAX_PLAUSIVEL = 20.0, 300.0
temps = longo["temperatura_c"].dropna()
fora = temps[(temps < MIN_PLAUSIVEL) | (temps > MAX_PLAUSIVEL)]
print()
print(f"Faixa observada: {temps.min():.2f} a {temps.max():.2f} C")
print(f"Leituras fora de [{MIN_PLAUSIVEL:.0f}, {MAX_PLAUSIVEL:.0f}] C: {len(fora)}")
if len(fora):
    print(fora.to_string())

## 8. Divisão nos 7 datasets

Esta é a segunda operação central. Para cada `Key Area`, montamos um dataset *wide* próprio e salvamos em CSV. A coluna `Key Area` não aparece dentro dos arquivos — ela é o critério que os separa, então deixa de ser uma variável dentro de cada modelo.

Os nulos são preservados: se um componente não tem leitura em determinado ensaio, a célula fica vazia (`NaN`), sem preenchimento ou média.


In [ ]:
ORDEM_FEATURES = ["blower_direction", "housing", "board_position", "run_case",
                  "losses", "deflector_hole", "material_coil_support"]

def slug(texto):
    return re.sub(r"[^a-z0-9]+", "_", texto.lower()).strip("_")

resumo = []
datasets_por_area = {}

for area, grupo in longo.groupby("key_area"):
    # Temperaturas: uma coluna por componente
    # dropna=False preserva componentes sem nenhuma medicao como coluna toda nula.
    # Sem isso, eles sumiriam da base em silencio.
    alvos = grupo.pivot_table(index="run_idx", columns="tc_name",
                              values="temperatura_c", aggfunc="first",
                              dropna=False)
    # Configurações de cada ensaio
    design = grupo.drop_duplicates("run_idx").set_index("run_idx")[ORDEM_FEATURES]
    wide = design.join(alvos).reset_index()

    nome_arquivo = f"dataset_{slug(area)}.csv"
    wide.to_csv(nome_arquivo, index=False)
    datasets_por_area[area] = wide

    n_componentes = alvos.shape[1]
    resumo.append({
        "key_area": area,
        "arquivo": nome_arquivo,
        "ensaios": wide.shape[0],
        "componentes": n_componentes,
    })

total_comp = sum(r["componentes"] for r in resumo)
print(f"Componentes distribuidos: {total_comp} (esperado: {len(componentes)})")
assert total_comp == len(componentes), "algum componente sumiu na divisao por area"

resumo = pd.DataFrame(resumo).sort_values("componentes", ascending=False)
resumo


## 9. Conferência final

Verificação rápida de que os 7 arquivos saíram com o número esperado de ensaios e componentes, e uma prévia de um deles.


In [ ]:
print("Datasets gerados:", len(datasets_por_area))
for area, d in datasets_por_area.items():
    print(f"  {area:16s} -> {d.shape[0]} ensaios x {d.shape[1]} colunas")


In [ ]:
# Prévia do dataset de Mainback
datasets_por_area["Mainback"].head()


---

### Resultado

Foram gerados **7 arquivos** `dataset_<area>.csv` mais o `limites_ativos.csv`.

Cada dataset está apenas **padronizado e dividido por área** — pronto para servir de base às próximas etapas (tratamento de outliers, nulos e codificação), que serão documentadas em notebooks separados.
